# Blue Book for Bulldozers — Exploratory Data Analysis

This notebook examines authorised Kaggle competition data before a model is fitted. It is deliberately unexecuted in Git because the licensed CSV files are not distributed with this repository. Put `TrainAndValid.csv` in `data/bluebook-for-bulldozers/` before running.

**Question:** What can historical auction records tell us about a bulldozer's sale price, without letting future dates leak into the past?


## Analysis plan

1. Verify the raw data contract and missingness.
2. Parse and inspect sale dates, price distribution, and time coverage.
3. Examine category/cardinality issues that affect preprocessing.
4. Record a temporal validation boundary before any model fitting.

Plots are descriptive. They do not establish that a machine characteristic causes its sale price.


In [ ]:
# Imports are intentionally limited to inspection tools. Parsing dates now
# prevents string sorting from silently producing an incorrect chronology.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = "data/bluebook-for-bulldozers/TrainAndValid.csv"
df = pd.read_csv(DATA_PATH, low_memory=False, parse_dates=["saledate"])
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
display(df.head())


## 1. Data quality and feature inventory

Auction records have substantial missingness and many categorical fields. We count missing values before choosing any imputation method; the modelling pipeline later learns imputers only from training-period rows.


In [ ]:
# Sorting is an EDA action and a modelling safeguard: every later temporal
# split assumes chronological order. Keep `saledate` until date features are made.
df = df.sort_values("saledate").copy()
quality = pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum(), "missing_pct": (df.isna().mean() * 100).round(2), "unique": df.nunique()}).sort_values("missing_pct", ascending=False)
display(quality)
print(f"Date range: {df.saledate.min().date()} to {df.saledate.max().date()}")


## 2. Price and time diagnostics

Sale price is positive and typically right-skewed, which is why the competition uses RMSLE. The time plot checks for changing market conditions; a random split would blend those conditions across train and validation.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.histplot(df["SalePrice"], bins=50, ax=axes[0], color="#5B3FD6")
axes[0].set(title="Auction sale-price distribution", xlabel="SalePrice")
monthly = df.set_index("saledate")["SalePrice"].resample("MS").median()
monthly.plot(ax=axes[1], color="#138A72")
axes[1].set(title="Monthly median sale price", ylabel="Median SalePrice")
fig.tight_layout()


## EDA decision record

The next notebook uses a strict date cutoff, date-derived features, training-fitted imputation, and unknown-aware ordinal encoding. This replaces manual category codes and manual test-column repair from the legacy notebook.
